# 📌 Hybrid AI Patterns

![Topic](https://img.shields.io/badge/Topic-Hybrid_AI_Patterns-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-Varia-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-November%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — The paper catalogues **13 reusable ways to couple a language model with an executable rule engine**, written in the style of the classic “Gang of Four” design patterns. The idea is to split the work:</span>
>
> * <span style="font-size:20px;">The model handles perception, language and ambiguity.</span>
> * <span style="font-size:20px;">The rules handle whatever must be guaranteed, audited or explained.</span>
>
> <span style="font-size:20px;">The patterns are held together by five invariants (model output is data, every answer is a dated observation, rules decide, evidence decides acceptance, failures stay distinct). The paper names five failure patterns that break those invariants, and it gives a decision guide for choosing a pattern. **The guide starts by admitting that a model alone is sometimes as good or better.**</span>

---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

## What it is about. 
Companies increasingly want AI to handle things like contracts, insurance claims, payments and patient intake, all of which come with strict rules. There are two basic ways to do it. You can paste the rules into the prompt and hope the model follows them. Or you can keep the rules as real, executable logic and use the model only for the things logic can’t do, like reading a document, phrasing a question, or suggesting a possibility.

In practice, teams have been inventing in-between solutions again and again, each under a different name: guardrails, agent frameworks, neuro-symbolic systems. The authors noticed the same handful of ideas keep turning up, so they gave each one a name, a description, and an honest list of strengths and weaknesses. They also report measurements for several of them, and the guide for choosing a pattern rests on pre-registered studies.

The ground rules are simple. Whatever the model says is treated as data, never as a command. Every answer is recorded, so you can replay and audit it. The rules make the actual decisions. A claim is accepted only on checkable evidence (like a quote that really appears in the source), not because the model sounded confident. And different kinds of failure, such as a crashed call, a badly formatted reply or an unclear answer, are never lumped together.

## How it is built. 
Every pattern is made from four moves: request (a rule asks the model or a tool), receipt (the answer returns as dated facts), accept (a rule decides whether it becomes a fact) and ask (a rule asks a person or asks for deciding evidence). The examples run on FrameX, a rule engine based on F-Logic, but the authors say any rule engine that can request a model call, store the answer as data and derive conclusions over it can host them.

## The four groups

|Group| Question it answers | Patterns |
|--|--------|------------|
| A. What crosses the boundary | What passes between model and rules? | 1 Model as Sensor, 2 Model as Author, 3 Model as Translator, 12 Model Selects, Rules Write |
| B. Who controls whom | Who decides and who constrains whom? | 4 Model Proposes, Rules Dispose; 5 Rules Steer the Model; 6 Critique and Repair; 7 Division by Competence; 13 Fail-Safe Union |
| C. The model in the reasoning | How does the model take part in the reasoning itself? | 8 Model as Hypothesis Generator, 9 Model as Critic of the Rules, 10 Rules Teach the Model |
| D. Over time | How is knowledge kept and revised? | 11 Rule-Governed Memory |

* **Group A (the boundary)** is about the content that moves between the two worlds. The model reads documents into proposed facts (1), drafts rules for checking (2), translates between a person’s words and the rules’ proofs (3), or only chooses passages and phrasing while the rules supply exact content (12).
* **Group B (control)** is about authority. The model can propose actions that rules admit or refuse (4). Rules can script the whole flow while the model only phrases and judges (5). Rules can drive bounded retries when a reply fails a check (6). Rules can handle what they cover and escalate the rest (7). Rules and models can be combined so that any source can only tighten the outcome (13).
* **Group C (reasoning)** puts the model inside the reasoning process, but never as the decider. It generates hypotheses that rules test (8), critiques rules where checks are incomplete (9), or is taught by rule-produced labels, examples and grades (10).
* **Group D (time)** has a single pattern. Memory across sessions is kept as statements that rules admit, supersede and withdraw, which makes “forget this” exact (11).

## The five failure patterns

These couplings recur in working systems and break one of the invariants. They resemble the good patterns, which is why they survive design review.

1. **Silent Rule Override.** Code outside the rules alters a verdict before the rules see it (for example a regex turning a judge’s “no” into “yes”). The outcome may be right, but the proof no longer contains the step that decided it. Fix: write the correction as a rule that reads the receipt, or, if it can only tighten, make it a source in a Fail-Safe Union.

2. **Circular Evaluation.** The cases or gold labels come from the model, prompt or developer being measured, or the criteria are chosen after seeing results. The numbers then measure agreement with oneself. Fix: fix cases, truths and criteria before the first model call, and derive labels by rules or construction.

3. **Self-Judge.** The model that wrote an answer, or one of the same family, judges whether it is right. Its errors and its judgement fail together. Fix: let rules check what can be checked and use a different model or prompt for the rest.

4. **Prose Policy.** The deciding policy exists only as sentences in a prompt. Nothing can test it, changes cannot be reviewed against cases, and an answer cannot show which sentence it followed. Fix: turn the policy into rules (Model as Author) and let Rules Steer the Model put only the relevant part into each prompt.

5. **Decorative Check.** A check is computed and logged, but nothing depends on it (a quoteFound no rule reads, a logged confidence). It looks like evidence and changes no outcome. Fix: have an acceptance rule read every check, and look for receipt fields with no outgoing arrow in the rule graph.

## When not to use a hybrid

The paper’s decision guide starts with this question, and it found four measured situations where a model alone was equal or better:

* **Few rules.** Up to about 48 questions per conversation, a reasoning model applying the rules from its prompt was not measurably worse than the hybrid. The accuracy break-even was around 64 questions.
* **Easy documents with a strong model.** On a clean five-document review, the model alone got 12 of 12 runs right, at lower cost (0.28 vs 0.72 cents) and in a third of the time.
* **Cost alone.** Once a provider’s cached-input discount is priced in, the hybrid’s cost advantage in conversations disappears.
* **A sensor that cannot quote.** A small 8B model did worse in the hybrid than alone (45% vs 60%), because a strict quote-checking rule refused correct values.

A hybrid is justified when at least one of these holds:

* The decision must come with a proof or audit trail.
* The model acts and some actions must never happen.
* Rules are many or the process is long (from about 64 rules in one conversation, in the authors’ runs).
* The answer aggregates over many objects (lists, counts, totals).
* A weaker, local or cheaper model has to do the work.
* The rules change and each change must be checked before release.

---
## 2. Patterns

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

## Group A: What crosses the boundary

### 2.1 Model as Sensor

**Definition**: The model reads what rules cannot (documents, speech, conversation answers) and turns it into proposed facts. A rule decides whether each proposal becomes a fact.

**How it works:** A rule fires a request for a pending document. The model answers with rows, each containing a value and a verbatim quote. The host measures whether the quote occurs in the source (quoteFound). An acceptance rule admits a statement only if the quote was found and that kind of document may make such a statement. Rejected statements keep their reason, and revoking a source withdraws everything that rested on it. A variant, Several Readers, One Rule, sends the same question to several models or samples. Agreement is accepted, and disagreement becomes “unclear” and is handed on.

**Use cases:**
* Contract and assignment review.
* Classifying listings from a fixed list.
* The “judge” in dialogue flows.
* Extracting memory claims.
* Any task that aggregates over many documents (lists, counts, totals).


| | Aspect |
|--|--------|
| 🟢 | **Every fact has provenance and a reason for acceptance.** |
| 🟢 | **Ungrounded answers stay rejected.** |
| 🟢 | **A reading exists once and is reused by many rules.** |
| 🟢 | **Aggregation was exact: 10/10 correct on 100 agreements, versus 0/6 exact exhaustive lists for a model alone.** |
| 🔴 | **One call per input and question. Extraction was expensive (43.6 cents once, versus 1.2 cents per question for the model alone).** |
| 🔴 | **A plausible but wrong value that passes every measurable check slips through.** |
| 🔴 | **A strict quote rule must fit the reader’s ability. An 8B model scored worse in the hybrid than alone (45% vs 60%).** |

### 2.2 Model as Author

**Definition**: The model writes formal material (rules, decision tables, schemas, tests, flows, thresholds). Every draft is checked offline before a person adopts it, and the engine never runs unchecked model output.

**How it works:** A policy text or failing case goes to the model, and the draft is stored as a receipt. A rule requests a check. The checker parses the draft, restricts which heads it may use, and evaluates it in a copy of the session against expectations and regression cases. It also reports which decisions would change. If the draft passes, it is recommended to an expert. If it fails, the checker’s message drives a repair (Pattern 6).

**Use cases:**
* Turning a policy sentence (“claims above 500 francs need a receipt”) into a rule.
* Updating a decision table after a regulation changes.
* Generating tests and dialogue flows.

| | Aspect |
|--|--------|
| 🟢 | **Writing becomes cheap and the cost moves to a complete check.** |
| 🟢 | **Out-of-language drafts are refused at the first bad token or cell.** |
| 🟢 | **The organisation keeps the adopted artefact, so the model is exchangeable.** |
| 🟢 | **5 of 5 drafting runs ended with a rule passing all hidden cases. Error messages that name the fix raised repairs within two attempts from 8/15 to 14/15.** |
| 🔴 | **Checking becomes the bottleneck, and the quality of error messages becomes a design parameter.** |
| 🔴 | **It needs a complete check for the faults that matter, regression cases written beforehand, and a person to adopt.** |
| 🔴 | **It cannot detect a draft that is well-formed, passes all cases, and is still wrong where no case covers it.** |



### 2.3 Model as Translator

**Definition**: The model sits between a person and the rules in both directions. It turns the person’s words into a request the rules can answer, and turns the rules’ answer and proof into words.

**How it works:** The model converts the question into a call to a read-only operation from a closed set (query, explain, why_not, preview, evidence_needs). Alternatively it writes a candidate query that is validated like any draft. The rules answer with a proof. The model phrases the answer, and a check binds the phrasing to the answer (the decision word, cited proof nodes). A variant, Read-Back Check, has an independent reader re-extract the facts from the written reply and compare them with what was meant to be stated.

**Use cases:**
* A clerk asking “can we pay this invoice?”
* Chat assistants that call a rules engine as a verified tool.
* Explanation layers over knowledge graphs.
* Exposing the rules as a read-only server to any model client.

| | Aspect |
|--|--------|
| 🟢 | **The person gets an answer in plain words with a proof behind it, including what evidence is missing.** |
| 🟢 | **A closed operation set keeps “model text is data” without qualification.** |
| 🟢 | **The fixed decision word prevents the model from contradicting the rules.** |
| 🔴 | **A mistranslated question yields a correct answer to the wrong question. Showing a back-translation to the user is the main safeguard.** |
| 🔴 | **Prose that adds facts not in the proof must be rejected, so the phrasing has to cite the proof.** |
| 🔴 | **Free queries inherit the limits of Model as Author: a query can validate and still ask the wrong thing.** |
| 🔴 | **On easy cases a strong model alone was as accurate and cheaper (12/12 runs; 0.28 vs 0.72 cents). Fixed templates beat a model where the wording is known.** |


### 2.4 Model Selects, Rules Write

**Definition**: The model chooses and phrases but never states content. It returns identifiers of passages the host numbered, and words around values it never sees. Every passage and value that reaches a person comes from the facts, word for word.

**How it works:** The host splits the source into numbered passages. The model answers with identifiers. A rule shows a passage only if the identifier names a real passage (the class is declared closed, so anything else is refused). A second request asks for words before and after a value without showing the value, and the rules insert the value between them, unless the words state a number of their own.

**Use cases:**
* Regulation Q&A where fees and required documents must be exact.
* Civil-protection alert generation with official hazard terms.
* Beneficial-ownership tools that cite line identifiers.
* Placeholder-based templates filled by code.

| | Aspect |
|--|--------|
| 🟢 | **It is correct by construction rather than by after-the-fact check. An invented requirement has no identifier, and a fee is never retyped.** |
| 🟢 | **Remaining errors are about relevance, not truth, and are visible in the source’s own words.** |
| 🟢 | **Hackathon figures, for a weaker form of the pattern (the model still saw the facts and code checked them afterwards): in alert generation, the official term and all facts appeared verbatim in 99% of 120 alerts against 45% for free translation (8B model).** |
| 🔴 | **The model can still pick the wrong or a missing passage.** |
| 🔴 | **It cannot combine or summarise across passages, and fluency is limited to the source’s.** |
| 🔴 | **It costs one call for selection and one for phrasing.** |
| 🔴 | **The evidence is hackathon figures only, not rerun by the authors. Selection does not close every gap (e.g. lower recall for 8B in anonymisation, 0.764 vs 0.894 for 70B).** |

## Group B: Who controls whom

### 2.5 Model Proposes, Rules Dispose

**Definition**: The model proposes an answer, action or choice. A rule admits it, refuses it with a reason, or narrows what may be proposed.

**How it works:** The proposal arrives as a receipt. Rules check capabilities, preconditions, limits and evidence requirements. A refusal goes back to the model with its reason so it can replan. A step bound stops a task whose proposals keep being refused. The host executes only what is admitted, and “task complete” is derived from recorded evidence (the model saying it is done is just another proposal). A variant, Simulator Disposes, lets a simulation’s receipt decide.

**Use cases:**
* Agents with tool calls, messages or transactions.
* Refunds above an authority limit.
* Portfolio changes below a materiality threshold.
* File writes outside the task scope.

| | Aspect |
|--|--------|
| 🟢 | **Effects become auditable and bounded.** |
| 🟢 | **The model keeps its initiative.** |
| 🟢 | **Conflicting constraints are reported to a person rather than silently prioritised.** |
| 🔴 | **The model may waste turns on refused proposals.** |
| 🔴 | **It bounds what can happen, not whether an admitted proposal is the best one.** |
| 🔴 | **It only covers failures the rule authors foresaw.** |
| 🔴 | **The paper has only a demonstration run for this pattern and no isolated measurement.** |

### 2.6 Rules Steer the Model

**Definition**: Rules decide what the model is asked, when, with which context and which tools. The model never decides the next step.

**How it works:** Each turn, rules derive the current step from the state. The dialogue model phrases that step’s question. After the person answers, a judge returns answered, not answered or unclear with a reason, and the rules pick the next step: branch, re-ask, hand over or end. An unanswered question never decides a branch. Rules also select which statements enter the prompt.

**Use cases:**
* Triage calls, interviews, intake and suitability flows.
* Customer support with hand-over.
* Agent phases and permissions.
* Point-in-time or jurisdiction-limited context.

| | Aspect |
|--|--------|
| 🟢 | **The protocol is executed completely regardless of length. The hybrid decided 770/770 runs correctly, while a model applying the rules itself fell to 39/60 at 64 questions and 31/60 at 96.** |
| 🟢 | **Each call is small, so a weaker model suffices (a 9B model worked in the hybrid; no model up to 70B did as a pure model).** |
| 🟢 | **Every turn has a receipt.** |
| 🔴 | **One or more calls per turn, and cost was about equal once cached input is priced.** |
| 🔴 | **A judge whose verdict contradicts its own reason must be caught by the rules, and verdict labels must not be readable as content.** |
| 🔴 | **With few rules (up to about 48 questions) a model alone was not measurably worse. In one triage head-to-head the model alone was more accurate until a judge fault was fixed.** |

### 2.7 Critique and Repair

**Definition**: When a reply fails a check, rules classify the failure, state it, and ask again with the problem and the faulty reply in the prompt. This happens a bounded number of times, and the loop stops on repetition.

**How it works:** Rules sort failures into six kinds:

* **Wrong format:** re-ask with the problem stated, up to a bound.
* **Not grounded:** reject the statement.
* **Not allowed:** refuse with a reason the model sees on its next turn.
* **Incomplete:** derive what is missing, and the answer becomes unknown.
* **Failed call:** stop with the reason, never treat it as “unclear”.
* **Plausible but wrong:** not detectable from the reply.

Each repair is a new request with its own receipt. Silent correction (mapping “ja” to yes by guessing) is excluded.

**Use cases:**
* Any pattern that checks a reply: format errors, ungrounded extractions, refused agent actions, failed regression cases in rule drafting.

| | Aspect |
|--|--------|
| 🟢 | **Repairs succeed when the message names the fix. First-request repairs rose from 5/10 to 10/10 in the paper’s study.** |
| 🟢 | **The bound keeps cost predictable.** |
| 🟢 | **It uses an external signal, avoiding the weakness of self-correction (back-prompting from a sound validator raised executable plans from about 12% to 82% in one cited planning study).** |
| 🔴 | **It cannot detect “plausible but wrong” replies, so a passed check is no protection against them.** |
| 🔴 | **It needs a validator that knows what is wrong, and every retry is an extra call.** |
| 🔴 | **Persistent failures simply stop with a reason, so escalation to a person or stronger model must be provided.** |

### 2.8 Division by Competence

**Definition**: Rules decide the cases they cover and send what remains open to a model or a person. Recurring answers are turned into rules over time.

**How it works:** The gate is the rules’ own three-valued result: true and false are decided, while unknown, unclear and conflict are handed on. The model’s verdict is only an assessment that travels with the case to a person and decides nothing. Recurring verdicts become candidate rules via Model as Author. The gate must never be the model’s own confidence.

**Use cases:**
* Claims processing, admissions, triage (“unclear” goes to a person).
* Evidence-gated casework with required human review.

| | Aspect |
|--|--------|
| 🟢 | **Model calls scale with the number of ambiguous cases, not all cases.** |
| 🟢 | **Proofs are kept wherever rules apply.** |
| 🟢 | **The rule part grows and the model part shrinks over time.** |
| 🔴 | **It needs high but incomplete rule coverage.** |
| 🔴 | **A person still decides the residue.** |
| 🔴 | **Recurring-answer distillation needs the authoring and checking machinery.** |
| 🔴 | **Demonstration run only, with no isolated measurement.** |

### 2.9 Fail-Safe Union

**Definition**: Verdicts from rules and models are combined under a declared order of outcomes where each source can only make the result stricter. A source that is silent, fails, or answers outside the order counts as the cautious outcome.

**How it works:** The outcome order is declared as facts (e.g. release < hold < block). Each source derives a verdict per case. A missing or unusable verdict gets the cautious verdict by rule. The highest-ranked verdict wins and the sources behind it are named (decidedBy). Variants: Strict union (failed call means cautious) and Rule floor (failed call falls back to the rules’ verdict, never less).

**Use cases:**
* Payment release and fraud checks.
* Personal-data redaction.
* Content retrieval with safety checks.
* In the spirit of “deny overrides” in access control.

| | Aspect |
|--|--------|
| 🟢 | **Adding a source can never weaken existing checks, and a model outage stops releases instead of letting them through.** |
| 🟢 | **Nothing is settled silently.** |
| 🟢 | **In an invoice agent (8B model), the strict union paid none of 300 invoices that must not be paid, versus 178 for the model alone and 89 for code checks alone.** |
| 🔴 | **Every source’s false alarms add up, so the hold rate grows with each source (here 7 of 200 legitimate invoices held, versus 2 for the model alone).** |
| 🔴 | **The guarantee breaks if any branch moves toward the loose side (an “unclear” or a failed call counted as agreement).** |
| 🔴 | **It is unsuitable where errors cost the same both ways, or where the strict outcome is itself harmful, such as a withheld emergency call.** |
| 🔴 | **It is stricter, not safe by itself. In redaction, 81% of documents still leaked at least one entity.** |
| 🔴 | **The evidence is hackathon figures only.** |

## Group C: The model inside the reasoning

### 2.10 Model as Hypothesis Generator

**Definition**: The model proposes what might be the case (a cause, a missing document, an explanation). Rules test each proposal in a copy of the session and ask for the evidence that decides between survivors. A hypothesis never becomes a fact.

**How it works:** Candidates arrive as rows marked hypothetical. Each is previewed in a copy of the session. Rules keep those that derive every observation without violating a constraint, and the evidence search names the one to four answers that would decide among them. A person or source supplies the answer.

**Use cases:**
* Bank reconciliation (fee? transposed amount? wrong booking date?).
* Diagnosis and investigation.
* Finding missing documents.

| | Aspect |
|--|--------|
| 🟢 | **Hypotheses are cheap to generate and never pollute the knowledge base.** |
| 🟢 | **The model contributes world knowledge while the rules supply exactness.** |
| 🟢 | **The pattern asks for the deciding evidence rather than guessing.** |
| 🔴 | **The search is bounded, and an incomplete search never proves that no explanation exists.** |
| 🔴 | **It needs constraints the rules can test exactly.** |
| 🔴 | **Demonstration run only, with no measurement.** |

### 2.11 Model as Critic of the Rules

**Definition**: The model reviews what the rules hold or conclude (a policy change, a rule against its source text, a surprising decision). Its critique opens a review and never changes a rule or conclusion.

**How it works:** The model receives the author’s rationale and a sealed before-and-after review, and answers with rows. A remark counts only if it names a case and quotes the rationale verbatim. Declared invariants are flagged without any model. Approval is derived only after a reviewer has disposed of every finding. A corrected rule proposed by the critic is checked like any draft.

**Use cases:**
* Pre-release review of policy changes.
* Fidelity of rules to cited legal text.
* Plausibility review of changed decisions.
* Anywhere checks are incomplete.

| | Aspect |
|--|--------|
| 🟢 | **It adds recall for semantic faults that regression cases miss.** |
| 🟢 | **Grounding filters out ungrounded remarks, and it never weakens a complete check.** |
| 🟢 | **In a pilot, the critic found the seeded contradiction in 29 of 30 reviews and made no remark on clean changes in 29 of 30.** |
| 🔴 | **False remarks cost reviewer time.** |
| 🔴 | **It missed a missing rule in most cases at 64 rules (found in only 3 of 10 rule reviews).** |
| 🔴 | **The pilot was small, with short changes and plainly stated defects.** |
| 🔴 | **It belongs on top of sound checks, never instead of them.** |

### 2.12 Rules Teach the Model

**Definition**: Rules supply labels, rule-selected worked examples and grades, with proofs, so that knowledge flows into the model’s context or weights. The rules stay in production as the independent check.

**How it works:** Rules label generated cases with their decision and missing requirements. The model answers each case, and a rule grades the answer against the label and records each gap it missed. Labels and grades leave the engine as training or evaluation data. After training, the same rules grade the taught model.

**Use cases:**
* Models that must work where the rules are not called (fast path, edge device).
* Teaching domain vocabulary.
* Trustworthy training and preference data, in the spirit of RLAIF and Constitutional AI.

| | Aspect |
|--|--------|
| 🟢 | **Labels come with proofs.** |
| 🟢 | **The independent rule check remains, so the model is never the authority.** |
| 🔴 | **Labels are only as good as the rules.** |
| 🔴 | **It needs a trainer outside the rule engine.** |
| 🔴 | **The paper has no measured model run yet, only a built setup with 144 generated cases.** |

## Group D: Over time

### 2.13 Rule-Governed Memory

**Definition**: What an assistant learns across conversations is kept as statements that rules admit, supersede and withdraw. Rules also decide which remembered statements enter the next prompt.

**How it works:** Each utterance triggers a Sensor request, and the proposed statements come back with quotes. A slot policy decides which sources may establish which facts (a chat can set a preference, but only a certificate a qualification). A newer admitted statement supersedes an older one without deleting it. A request to forget marks the utterance, and acceptance is recomputed, so everything resting on it loses acceptance (an older statement may become current again). Before each call a steering rule selects the statements to include, and the reply cites their identifiers. Rules check that each cited statement was sent and is current.

**Use cases:**
* Advisors or assistants serving the same person across sessions.
* Data that changes over time.
* Exact “forget this” requests.

| | Aspect |
|--|--------|
| 🟢 | **Every remembered value has an utterance, quote, date and history.** |
| 🟢 | **Forgetting is exact, with no manual clean-up.** |
| 🟢 | **Chat cannot establish what its source may not, and derived facts (such as an admission) are never written to memory.** |
| 🔴 | **One Sensor call per utterance, and a store that grows (only forgetting removes content).** |
| 🔴 | **A wrongly admitted statement holds until superseded or forgotten.** |
| 🔴 | **Conflicts between sources need a person.** |
| 🔴 | **Similarity retrieval stays outside the rules.** |
| 🔴 | **Demonstration only, with no measurement.** |

## 6. Source


Siegfried Handschuh, Götz-Henrik Wiegand, Lorena Raichle (2026). *Hybrid AI Patterns: Thirteen Ways to Couple Language Models with Rules, and How to Choose Among Them*. arXiv:[2610.40036](https://arxiv.org/abs/2610.40036)